# PyTorch: Zero to Hero — Building Transformers

> A hands-on training notebook covering everything you need to build, train, and optimize Transformer models with PyTorch — from tensors to full architecture implementation.

---

## Table of Contents
1. [PyTorch Fundamentals: Tensors and Operations](#1)
2. [Autograd and Backpropagation](#2)
3. [Building Neural Networks with nn.Module](#3)
4. [Optimizers and Loss Functions](#4)
5. [Custom Dataset and DataLoader](#5)
6. [Embedding Layers and Positional Encoding](#6)
7. [Scaled Dot-Product Attention](#7)
8. [Multi-Head Attention](#8)
9. [Feed-Forward Networks and Layer Normalization](#9)
10. [Encoder and Decoder Blocks](#10)
11. [Full Transformer Architecture](#11)
12. [Training Loop and Performance Optimization](#12)
13. [Mixed Precision Training with torch.cuda.amp](#13)
14. [Model Checkpointing and Saving](#14)


---
<a id="1"></a>
## Section 1 — PyTorch Fundamentals: Tensors and Operations

Tensors are the core data structure in PyTorch — analogous to NumPy arrays but with GPU acceleration and autograd support. Everything in a neural network is ultimately a tensor operation.

**Key concepts covered:**
- Tensor creation: `torch.tensor`, `torch.zeros`, `torch.ones`, `torch.randn`
- Shapes, dtypes, and devices
- Indexing, slicing, and reshaping
- Matrix multiplication and broadcasting


In [1]:
import torch
import torch.nn as nn
import math

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


PyTorch version: 2.8.0+cu128
CUDA available: False
Using device: cpu


In [2]:
# ── Tensor Creation ──────────────────────────────────────────────────────────
t_manual  = torch.tensor([[1.0, 2.0], [3.0, 4.0]])          # from data
t_zeros   = torch.zeros(3, 4)                                 # all zeros  shape (3,4)
t_ones    = torch.ones(2, 5)                                  # all ones   shape (2,5)
t_randn   = torch.randn(2, 3)                                 # N(0,1)     shape (2,3)
t_arange  = torch.arange(0, 10, 2)                           # [0,2,4,6,8]
t_linspace= torch.linspace(0, 1, steps=5)                    # [0,.25,.5,.75,1]

print("manual:\n",  t_manual)
print("zeros:\n",   t_zeros)
print("randn:\n",   t_randn)
print("arange:",    t_arange)
print("linspace:",  t_linspace)


manual:
 tensor([[1., 2.],
        [3., 4.]])
zeros:
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
randn:
 tensor([[ 0.4298,  0.8859, -1.3070],
        [-0.9457, -0.7235, -0.0086]])
arange: tensor([0, 2, 4, 6, 8])
linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


In [ ]:
# ── Shapes, dtypes, and devices ──────────────────────────────────────────────
x = torch.randn(4, 6, dtype=torch.float32)

print("shape  :", x.shape)          # torch.Size([4, 6])
print("ndim   :", x.ndim)           # 2
print("dtype  :", x.dtype)          # torch.float32
print("device :", x.device)         # cpu

# Cast dtype
x_int  = x.to(torch.int32)
x_half = x.to(torch.float16)
print("int32  :", x_int.dtype)
print("float16:", x_half.dtype)

# Move to GPU (no-op if not available)
x_gpu = x.to(DEVICE)
print("on device:", x_gpu.device)


In [ ]:
# ── Indexing, Slicing, Reshaping ─────────────────────────────────────────────
t = torch.arange(24).reshape(2, 3, 4)   # shape (2, 3, 4)
print("tensor:\n", t)
print("t[0]:\n",    t[0])               # first "batch"
print("t[:, 1, :]:\n", t[:, 1, :])     # all batches, row 1
print("t[..., -1]:\n", t[..., -1])     # last column of every 2-D slice

# Common reshape ops
flat    = t.view(-1)             # flatten → (24,)
reshaped= t.reshape(6, 4)        # (6, 4)
permuted= t.permute(0, 2, 1)     # swap last two dims → (2, 4, 3)
squeezed= torch.randn(1, 4, 1).squeeze()   # remove size-1 dims → (4,)
unsqueezed = flat.unsqueeze(0)   # add dim at pos 0 → (1, 24)

print("\nflat:", flat.shape, "| reshaped:", reshaped.shape,
      "| permuted:", permuted.shape, "| squeezed:", squeezed.shape)


In [ ]:
# ── Matrix Multiplication and Broadcasting ───────────────────────────────────
A = torch.randn(3, 4)
B = torch.randn(4, 5)

# Matrix multiply:  (3,4) @ (4,5) → (3,5)
C = torch.matmul(A, B)           # or A @ B
print("matmul (3,4)@(4,5) →", C.shape)

# Batched matmul:   (8, 3, 4) @ (8, 4, 5) → (8, 3, 5)
A_batch = torch.randn(8, 3, 4)
B_batch = torch.randn(8, 4, 5)
C_batch = torch.bmm(A_batch, B_batch)
print("bmm (8,3,4)@(8,4,5) →", C_batch.shape)

# torch.einsum — expressive notation used heavily in attention
# Equivalent to batched matmul: b=batch, i=seq, j=head_dim, k=seq
D = torch.einsum("bij,bjk->bik", A_batch, B_batch)
print("einsum result:", D.shape)

# Broadcasting  — PyTorch aligns dimensions from the right
bias = torch.ones(5)             # (5,)
out  = C + bias                  # (3,5) + (5,) → (3,5)  [broadcast]
print("broadcast (3,5) + (5,) →", out.shape)

# Common element-wise ops
print("\nsum:", A.sum())
print("mean row:", A.mean(dim=1))   # mean over columns → (3,)
print("max along dim0:", A.max(dim=0).values)
print("softmax:\n", torch.softmax(A[0], dim=-1))


### 🏋️ Exercises — Section 1: Tensors and Operations

**E1.1** Create a tensor of shape `(5, 5)` filled with values from 1 to 25 (use `torch.arange` + `reshape`). Then extract the diagonal using `torch.diag`.

**E1.2** Create two random tensors `A` of shape `(4, 3)` and `B` of shape `(3, 6)`. Compute their matrix product using both `torch.matmul` and `torch.einsum`. Verify the results are equal.

**E1.3** Without any loops, compute the **pairwise squared Euclidean distance** between all rows of a `(16, 8)` tensor `X`. The result should be shape `(16, 16)`. *(Hint: expand `||a - b||² = ||a||² + ||b||² - 2a·bᵀ`)*

**E1.4** Given a `(batch=4, seq=10, dim=32)` tensor, apply softmax over the `seq` dimension, then verify each row sums to 1.


In [ ]:
# ── E1.1 ─────────────────────────────────────────────────────────────────────
# YOUR CODE HERE
# t = ...
# diag = ...

# ── E1.2 ─────────────────────────────────────────────────────────────────────
# YOUR CODE HERE
# A = torch.randn(4, 3)
# B = torch.randn(3, 6)
# result_matmul = ...
# result_einsum = ...
# print(torch.allclose(result_matmul, result_einsum))

# ── E1.3 — Pairwise squared distance ─────────────────────────────────────────
# YOUR CODE HERE
# X = torch.randn(16, 8)
# dists = ...   # shape (16, 16)

# ── E1.4 — Softmax over seq dim ───────────────────────────────────────────────
# YOUR CODE HERE
# x = torch.randn(4, 10, 32)
# s = ...   # softmax over dim=1
# print(s.sum(dim=1))  # should be all ones


---
<a id="2"></a>
## Section 2 — Autograd and Backpropagation

PyTorch builds a **dynamic computational graph** at runtime. When you call `.backward()`, it traverses this graph in reverse to compute gradients via the chain rule.

**Key concepts:**
- `requires_grad=True` — opt a tensor into gradient tracking
- `loss.backward()` — populate `.grad` on all leaf tensors
- `optimizer.zero_grad()` — reset gradients before each update (prevent accumulation)
- `torch.no_grad()` — context manager to disable gradient tracking (e.g., during inference)


In [ ]:
# ── requires_grad and the computational graph ────────────────────────────────
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# Forward pass:  y = w*x + b  → loss = y^2
y    = w * x + b          #  3*2 + 1 = 7
loss = y ** 2             #  7^2     = 49

print(f"y = {y.item():.1f}  |  loss = {loss.item():.1f}")
print(f"y.grad_fn = {y.grad_fn}")        # MulBackward / AddBackward
print(f"loss.grad_fn = {loss.grad_fn}")  # PowBackward

# Backward pass — compute gradients
loss.backward()

# d(loss)/dx = d(y^2)/dx = 2y * w = 2*7*3 = 42
print(f"\nd(loss)/dw = {w.grad.item():.1f}")  # 2*y*x  = 2*7*2 = 28
print(f"d(loss)/dx = {x.grad.item():.1f}")  # 2*y*w  = 2*7*3 = 42
print(f"d(loss)/db = {b.grad.item():.1f}")  # 2*y*1  = 14


In [ ]:
# ── Gradient accumulation gotcha and zero_grad ───────────────────────────────
w = torch.tensor(1.0, requires_grad=True)

for step in range(3):
    loss = (w * 3) ** 2
    loss.backward()
    print(f"step {step}  w.grad = {w.grad.item():.1f}  (accumulated!)")

# Gradients stack up — always zero before the next backward
print("\n--- resetting with zero_grad ---")
w.grad = None   # or optimizer.zero_grad()

for step in range(3):
    loss = (w * 3) ** 2
    loss.backward()
    print(f"step {step}  w.grad = {w.grad.item():.1f}")
    w.grad = None   # reset after each step

# ── torch.no_grad for inference ───────────────────────────────────────────────
print("\nInference mode (no graph built):")
with torch.no_grad():
    out = w * 5
    print(f"  out.requires_grad = {out.requires_grad}")   # False


### 🏋️ Exercises — Section 2: Autograd and Backpropagation

**E2.1** Implement **gradient descent by hand** for 50 steps to minimise `f(x) = (x - 3)²`. Start at `x = 0.0`. Use `requires_grad=True`, manually subtract `lr * x.grad` each step (do NOT use an optimizer), and print `x` every 10 steps.

**E2.2** Define `z = x³ + 2x² - 5x` and compute `dz/dx` at `x = 2.0` using autograd. Verify by computing the derivative analytically: `dz/dx = 3x² + 4x - 5`.

**E2.3** Write a function `numerical_gradient(f, x, eps=1e-5)` that computes the gradient of scalar-output function `f` at tensor `x` using the **finite difference approximation**: `(f(x+eps) - f(x-eps)) / (2*eps)`. Compare its result to `autograd` for `f(x) = sum(sin(x²))` on a `(3,)` tensor.


In [ ]:
# ── E2.1 — Manual gradient descent ──────────────────────────────────────────
lr = 0.1
x = torch.tensor(0.0, requires_grad=True)
# YOUR CODE HERE: run 50 GD steps, print x every 10 steps

# ── E2.2 — Autograd vs analytical derivative ─────────────────────────────────
x = torch.tensor(2.0, requires_grad=True)
# YOUR CODE HERE
# z = ...
# z.backward()
# autograd_grad = x.grad.item()
# analytical_grad = ...
# print(f"autograd: {autograd_grad}  |  analytical: {analytical_grad}")

# ── E2.3 — Numerical gradient check ─────────────────────────────────────────
def numerical_gradient(f, x, eps=1e-5):
    # YOUR CODE HERE
    grad = torch.zeros_like(x)
    for i in range(x.numel()):
        pass   # replace with finite-difference computation
    return grad

x = torch.tensor([1.0, 2.0, 3.0])
# Compare numerical_gradient result to autograd result for f(x) = sum(sin(x**2))
# YOUR CODE HERE


---
<a id="3"></a>
## Section 3 — Building Neural Networks with `nn.Module`

`nn.Module` is the base class for every layer and model in PyTorch. It automatically:
- registers parameters (`.parameters()`) for the optimizer
- manages device placement (`.to(device)`)
- toggles between train/eval modes (`.train()` / `.eval()`)

**Building blocks used in Transformers:**
| Module | Purpose |
|---|---|
| `nn.Linear(in, out)` | Learnable weight matrix + optional bias |
| `nn.Embedding(vocab, dim)` | Lookup table for token embeddings |
| `nn.LayerNorm(dim)` | Normalizes over the last `dim` features |
| `nn.Dropout(p)` | Randomly zeroes activations during training |
| `nn.Sequential(...)` | Chains modules without writing a forward() |


In [ ]:
# ── Custom nn.Module ─────────────────────────────────────────────────────────
class SimpleMLP(nn.Module):
    """A 2-layer MLP: Linear → ReLU → Linear"""

    def __init__(self, in_features: int, hidden: int, out_features: int):
        super().__init__()
        # All nn.Module or nn.Parameter objects assigned here are *registered*
        self.fc1 = nn.Linear(in_features, hidden)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden, out_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Called when you do  model(x)
        return self.fc2(self.act(self.fc1(x)))


model = SimpleMLP(in_features=16, hidden=64, out_features=8)
x_in  = torch.randn(4, 16)   # batch of 4

print("Forward output shape:", model(x_in).shape)

# Inspect parameters
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params    : {total_params:,}")
print(f"Trainable params: {trainable:,}")

print("\nNamed parameters:")
for name, param in model.named_parameters():
    print(f"  {name:20s}  {tuple(param.shape)}")


In [ ]:
# ── nn.Sequential shorthand ──────────────────────────────────────────────────
seq_model = nn.Sequential(
    nn.Linear(16, 64),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(64, 8),
)
print("Sequential output:", seq_model(x_in).shape)

# ── train / eval mode affects Dropout and BatchNorm ──────────────────────────
seq_model.train()   # Dropout is ACTIVE
out_train = seq_model(x_in)

seq_model.eval()    # Dropout is DISABLED — outputs are deterministic
with torch.no_grad():
    out_eval = seq_model(x_in)

# In training mode outputs will vary; in eval mode they are stable
print(f"Train mode output[0,:3]: {out_train[0,:3].detach()}")
print(f"Eval  mode output[0,:3]: {out_eval[0,:3]}")


### 🏋️ Exercises — Section 3: Building with nn.Module

**E3.1** Build a `ResidualBlock` module: `output = x + Linear(ReLU(Linear(x)))` where both linears map `d → d`. Verify that the output shape equals the input shape for a `(4, 64)` input.

**E3.2** Add a `__repr__` method to `SimpleMLP` so that `print(model)` shows the architecture in a structured way (beyond the default PyTorch repr).

**E3.3** Build a `ParameterNormLinear` module that has a single learnable weight matrix `W` (registered as `nn.Parameter`), and normalises each **output** neuron's weights to unit norm before the forward pass. Use `F.normalize`. *(This is the basis of weight normalization.)*

**E3.4** Count the number of **non-trainable** (frozen) parameters in a model. Then freeze all parameters in `fc1` of a `SimpleMLP` and verify only `fc2` parameters are updated after one optimizer step.


In [ ]:
import torch.nn.functional as F

# ── E3.1 — ResidualBlock ──────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        # YOUR CODE HERE

    def forward(self, x):
        # YOUR CODE HERE: return x + Linear(ReLU(Linear(x)))
        pass

# Test
# block = ResidualBlock(64)
# out = block(torch.randn(4, 64))
# assert out.shape == (4, 64)
# print("ResidualBlock output:", out.shape)

# ── E3.2 — Custom __repr__ ───────────────────────────────────────────────────
# Add __repr__ to SimpleMLP (or a new subclass) — YOUR CODE HERE

# ── E3.3 — ParameterNormLinear ────────────────────────────────────────────────
class ParameterNormLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = nn.Parameter(torch.randn(out_features, in_features))
        # YOUR CODE HERE: add bias if desired

    def forward(self, x):
        # Normalize each row of W to unit norm, then do x @ W.T
        # YOUR CODE HERE
        pass

# ── E3.4 — Freeze fc1 and verify only fc2 updates ────────────────────────────
# YOUR CODE HERE
# model = SimpleMLP(16, 64, 8)
# Freeze fc1
# for p in model.fc1.parameters():
#     p.requires_grad = ...
# Count frozen params
# Run one optimizer step and confirm fc1.weight hasn't changed


---
<a id="4"></a>
## Section 4 — Optimizers and Loss Functions

A typical **training step** follows this pattern every batch:

```
optimizer.zero_grad()   # 1. clear stale gradients
output = model(x)       # 2. forward pass
loss = criterion(output, target)  # 3. compute loss
loss.backward()         # 4. backprop
optimizer.step()        # 5. update weights
```

**Common loss functions:**
| Loss | Use case |
|---|---|
| `nn.CrossEntropyLoss` | Multi-class classification / language modelling |
| `nn.MSELoss` | Regression |
| `nn.BCEWithLogitsLoss` | Binary classification |

**Common optimizers:**
| Optimizer | Notes |
|---|---|
| `SGD` | Simple; momentum variant often used |
| `Adam` | Adaptive LR; default choice |
| `AdamW` | Adam + decoupled weight decay — **standard for Transformers** |


In [ ]:
import torch.optim as optim

# ── Loss Functions ────────────────────────────────────────────────────────────
# CrossEntropyLoss:  logits shape (N, C),  targets shape (N,) with class indices
logits  = torch.randn(8, 10)         # 8 samples, 10 classes
targets = torch.randint(0, 10, (8,)) # true class indices

ce_loss = nn.CrossEntropyLoss()
print(f"CrossEntropy loss: {ce_loss(logits, targets).item():.4f}")

# MSELoss
pred   = torch.randn(8, 1)
gt     = torch.randn(8, 1)
mse    = nn.MSELoss()
print(f"MSE loss         : {mse(pred, gt).item():.4f}")

# ── Optimizers ────────────────────────────────────────────────────────────────
model = SimpleMLP(16, 64, 10).to(DEVICE)

# AdamW — standard for Transformers
#   weight_decay separates L2 penalty from the adaptive LR mechanism
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

# ── One Full Training Step ────────────────────────────────────────────────────
x      = torch.randn(8, 16).to(DEVICE)
labels = torch.randint(0, 10, (8,)).to(DEVICE)

model.train()
optimizer.zero_grad()               # step 1
output = model(x)                   # step 2  — shape (8, 10)
loss   = ce_loss(output, labels)    # step 3
loss.backward()                     # step 4
optimizer.step()                    # step 5

print(f"\nAfter one step  loss = {loss.item():.4f}")
print(f"fc1.weight.grad norm: {model.fc1.weight.grad.norm().item():.4f}")


### 🏋️ Exercises — Section 4: Optimizers and Loss Functions

**E4.1** Implement **label smoothing** manually: given one-hot targets, replace `1.0` with `1 - ε` and distribute `ε / (C - 1)` to all other classes (`ε = 0.1`, `C = 10`). Compute cross-entropy loss with the smoothed targets using `nn.KLDivLoss`.

**E4.2** Compare convergence of **SGD**, **SGD+Momentum**, and **AdamW** on fitting a `SimpleMLP` to random regression data for 200 steps. Plot (or print) the loss every 20 steps.

**E4.3** Implement the **Transformer paper's LR schedule**: `lr = d_model^(-0.5) * min(step^(-0.5), step * warmup^(-1.5))`. Plot its shape for 10 000 steps with `d_model=512, warmup=4000`.

**E4.4** Implement **gradient accumulation** — simulate a logical batch of 32 by running 4 forward-backward passes with `batch_size=8` before a single optimizer step. Confirm the final gradient is the same as a single pass with `batch_size=32`.


In [ ]:
# ── E4.1 — Label Smoothing ───────────────────────────────────────────────────
eps, C = 0.1, 10
logits  = torch.randn(8, C)
targets = torch.randint(0, C, (8,))
# Smooth the targets into soft distributions
# YOUR CODE HERE
# smooth_targets = ...
# loss = nn.KLDivLoss(reduction="batchmean")(F.log_softmax(logits, dim=-1), smooth_targets)
# print(f"Label-smoothed loss: {loss.item():.4f}")

# ── E4.2 — Compare optimizers ────────────────────────────────────────────────
# YOUR CODE HERE
# Build a small regression dataset (X: 64x8, y: 64x1)
# Train with SGD, SGD+momentum, AdamW for 200 steps each
# Print loss every 20 steps for each

# ── E4.3 — Transformer LR schedule ──────────────────────────────────────────
def transformer_lr(step, d_model=512, warmup=4000):
    # YOUR CODE HERE
    pass

# steps = range(1, 10001)
# lrs = [transformer_lr(s) for s in steps]
# for s in [100, 500, 1000, 4000, 8000, 10000]:
#     print(f"step {s:>6}: lr = {transformer_lr(s):.6f}")

# ── E4.4 — Gradient accumulation ─────────────────────────────────────────────
# YOUR CODE HERE
# Verify grad with accumulation (4 * bs=8) == single forward pass (bs=32)
# Hint: set the same manual seed before each approach; use torch.manual_seed(42)


---
<a id="5"></a>
## Section 5 — Custom Dataset and DataLoader

PyTorch separates **data layout** (`Dataset`) from **data loading mechanics** (`DataLoader`). This is what you implement for any Transformer training pipeline:

- `Dataset.__len__()` — returns the dataset size
- `Dataset.__getitem__(idx)` — returns one sample `(input_ids, target_ids)`
- `DataLoader` — handles batching, shuffling, and parallel workers automatically


In [ ]:
from torch.utils.data import Dataset, DataLoader

# ── Custom Sequence Dataset (e.g. for a language model) ──────────────────────
class SequenceDataset(Dataset):
    """
    Synthetic dataset: each sample is a random token sequence.
    Input = tokens[0 : seq_len-1],  Target = tokens[1 : seq_len]
    (the classic next-token prediction setup used to train GPT-style models)
    """

    def __init__(self, num_samples: int, seq_len: int, vocab_size: int):
        super().__init__()
        self.data = torch.randint(0, vocab_size, (num_samples, seq_len))

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        seq = self.data[idx]
        return seq[:-1], seq[1:]   # (input_ids, target_ids)


# ── DataLoader ────────────────────────────────────────────────────────────────
VOCAB_SIZE  = 256
SEQ_LEN     = 32
BATCH_SIZE  = 8

dataset    = SequenceDataset(num_samples=1000, seq_len=SEQ_LEN, vocab_size=VOCAB_SIZE)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Peek at one batch
inp, tgt = next(iter(dataloader))
print(f"Dataset length     : {len(dataset)}")
print(f"Batches per epoch  : {len(dataloader)}")
print(f"Input  batch shape : {inp.shape}")   # (8, 31)
print(f"Target batch shape : {tgt.shape}")   # (8, 31)
print(f"Sample input  ids  : {inp[0, :8]}")
print(f"Sample target ids  : {tgt[0, :8]}  (shifted by 1)")


### 🏋️ Exercises — Section 5: Dataset and DataLoader

**E5.1** Create a `BucketBatchSampler` that groups sequences by length into buckets and builds batches with minimal padding. For simplicity, sort the dataset by sequence length and yield indices in sorted order. Wrap it in a `DataLoader` using `batch_sampler=`.

**E5.2** Implement a `collate_fn` for variable-length sequences: given a list of `(input_ids, target_ids)` tuples with different lengths, pad them to the same length using `torch.nn.utils.rnn.pad_sequence` and return a padding mask tensor.

**E5.3** Add a **data augmentation** transform to `SequenceDataset`: randomly replace 15% of input token ids with a special `[MASK]` token id (e.g. `vocab_size - 1`). This mimics BERT-style masked language modeling. Verify the masking ratio is approximately correct.


In [ ]:
from torch.utils.data import Sampler
from torch.nn.utils.rnn import pad_sequence

# ── E5.1 — BucketBatchSampler ────────────────────────────────────────────────
class BucketBatchSampler(Sampler):
    """Yields batches of indices grouped by sequence length."""

    def __init__(self, dataset, batch_size: int):
        self.batch_size = batch_size
        # Sort indices by sequence length (use len of first element of each sample)
        # YOUR CODE HERE
        self.sorted_indices = list(range(len(dataset)))  # replace with sorted version

    def __iter__(self):
        # YOUR CODE HERE: yield chunks of self.sorted_indices of size self.batch_size
        pass

    def __len__(self):
        return (len(self.sorted_indices) + self.batch_size - 1) // self.batch_size

# ── E5.2 — Collate with padding ──────────────────────────────────────────────
def padded_collate(batch):
    """
    batch: list of (input_ids, target_ids) — variable length tensors
    Returns: padded inputs, padded targets, and a boolean padding mask
    """
    inputs, targets = zip(*batch)
    # YOUR CODE HERE
    # padded_inputs  = pad_sequence(...)   # left-pads or right-pads
    # padded_targets = pad_sequence(...)
    # mask = ...   # True where padding
    pass

# ── E5.3 — Masked LM Dataset augmentation ────────────────────────────────────
MASK_TOKEN = VOCAB_SIZE - 1
MASK_PROB  = 0.15

class MaskedSequenceDataset(SequenceDataset):
    """Randomly mask 15% of input tokens (BERT-style)."""

    def __getitem__(self, idx: int):
        inp, tgt = super().__getitem__(idx)
        # YOUR CODE HERE: randomly replace MASK_PROB fraction of inp with MASK_TOKEN
        # masked_inp = inp.clone()
        # mask = ...
        # masked_inp[mask] = MASK_TOKEN
        # return masked_inp, inp   # (masked input, original as target)
        return inp, tgt  # replace this line

# Verify ~15% masking
# ds = MaskedSequenceDataset(100, SEQ_LEN, VOCAB_SIZE)
# inp, tgt = ds[0]
# frac_masked = (inp == MASK_TOKEN).float().mean().item()
# print(f"Masking fraction: {frac_masked:.2f}  (expected ~{MASK_PROB})")


---
<a id="6"></a>
## Section 6 — Embedding Layers and Positional Encoding

Transformers have **no recurrence** — they process all positions simultaneously. Therefore the model needs an explicit signal about *where* each token is. The original "Attention is All You Need" paper uses **sinusoidal positional encoding**:

$$PE(pos,\, 2i)   = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE(pos,\, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

The encoding is **added** to the token embedding, giving the model both semantic and positional information in the same vector space.


In [ ]:
class TokenEmbedding(nn.Module):
    """Scales token embeddings by sqrt(d_model) as done in the original paper."""

    def __init__(self, vocab_size: int, d_model: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.scale = math.sqrt(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len)  →  (batch, seq_len, d_model)
        return self.embed(x) * self.scale


class PositionalEncoding(nn.Module):
    """
    Fixed sinusoidal positional encoding (non-learnable).
    Registered as a buffer — moves with the model to any device but is not
    a trainable parameter.
    """

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Build the PE table  shape (max_len, d_model)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()          # (max_len, 1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )                                                              # (d_model/2,)

        pe[:, 0::2] = torch.sin(pos * div)   # even indices
        pe[:, 1::2] = torch.cos(pos * div)   # odd  indices

        pe = pe.unsqueeze(0)          # (1, max_len, d_model) — broadcast over batch
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


# ── Quick sanity check ────────────────────────────────────────────────────────
D_MODEL = 128
tok_emb = TokenEmbedding(VOCAB_SIZE, D_MODEL)
pos_enc = PositionalEncoding(D_MODEL, max_len=SEQ_LEN)

token_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN - 1))
embed_out  = tok_emb(token_ids)     # (8, 31, 128)
final_emb  = pos_enc(embed_out)     # (8, 31, 128)

print(f"Token embedding output : {embed_out.shape}")
print(f"After positional enc   : {final_emb.shape}")
print(f"PE table shape         : {pos_enc.pe.shape}")


### 🏋️ Exercises — Section 6: Embeddings and Positional Encoding

**E6.1** Implement **Rotary Position Embedding (RoPE)** — the positional scheme used in LLaMA. For a head dimension `d`, split the vector into pairs and rotate each pair by angle `θ_i = pos / 10000^(2i/d)`. Apply it to a single query vector.

**E6.2** Implement **learnable positional encoding**: replace the fixed sinusoidal table in `PositionalEncoding` with an `nn.Embedding(max_len, d_model)` that is trained via backprop. Compare the number of trainable parameters.

**E6.3** Visualise the sinusoidal PE table for `d_model=64, max_len=50` as a heatmap using `matplotlib` (or just print it). Observe how lower dimensions complete full cycles at shorter positions — this is what gives the model multi-resolution positional structure.

**E6.4** Implement **relative positional bias**: instead of absolute PE, add a learnable scalar bias `b[i-j]` to the attention score between position `i` and position `j`. The bias table should have shape `(2 * max_len - 1,)`.


In [ ]:
# ── E6.1 — RoPE (Rotary Position Embedding) ──────────────────────────────────
def apply_rope(x: torch.Tensor, pos: int) -> torch.Tensor:
    """
    Apply RoPE to a single vector x of shape (d,).
    Split into pairs (x[0],x[1]), (x[2],x[3]), ...
    Rotate each pair (a,b) by angle theta_i:
        [a*cos(θ) - b*sin(θ),  a*sin(θ) + b*cos(θ)]
    """
    d = x.shape[-1]
    assert d % 2 == 0
    # YOUR CODE HERE
    return x  # replace with rotated version

# ── E6.2 — Learnable positional encoding ─────────────────────────────────────
class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        # YOUR CODE HERE: use nn.Embedding instead of a fixed buffer

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

# Compare param counts
fixed_pe    = PositionalEncoding(D_MODEL, SEQ_LEN)
learnable_pe= LearnablePositionalEncoding(D_MODEL, SEQ_LEN)
# print("Fixed PE params    :", sum(p.numel() for p in fixed_pe.parameters()))
# print("Learnable PE params:", sum(p.numel() for p in learnable_pe.parameters()))

# ── E6.3 — Visualise the PE table ────────────────────────────────────────────
# YOUR CODE HERE
# pe_table = PositionalEncoding(64, max_len=50)
# table = pe_table.pe.squeeze(0).detach().numpy()  # shape (50, 64)
# Try: import matplotlib.pyplot as plt; plt.imshow(table, aspect="auto"); plt.show()
# Or just: print(table[:5, :8].round(2))

# ── E6.4 — Relative positional bias ─────────────────────────────────────────
class RelativePositionBias(nn.Module):
    def __init__(self, max_len: int):
        super().__init__()
        # bias table covers offsets from -(max_len-1) to +(max_len-1)
        # YOUR CODE HERE: self.bias = nn.Parameter(...)

    def forward(self, seq_len: int) -> torch.Tensor:
        # Return a (seq_len, seq_len) matrix of biases
        # YOUR CODE HERE
        pass


---
<a id="7"></a>
## Section 7 — Scaled Dot-Product Attention

The core operation that gives Transformers their power:

$$\text{Attention}(Q,\, K,\, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** (query), **K** (key), **V** (value) — all linear projections of the same input (self-attention) or different inputs (cross-attention)  
- **Scaling by $\sqrt{d_k}$** — prevents extremely small gradients when $d_k$ is large (dot products grow with dimension)
- **Mask** — set forbidden positions to $-\infty$ before softmax so they get zero attention weight  
  - *Padding mask* — ignore `[PAD]` tokens  
  - *Causal (look-ahead) mask* — prevent position $i$ from attending to positions $> i$ (used in decoder)


In [ ]:
def scaled_dot_product_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    mask: torch.Tensor | None = None,
    dropout: float = 0.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Scaled dot-product attention.

    Args:
        Q: (..., seq_q, d_k)
        K: (..., seq_k, d_k)
        V: (..., seq_k, d_v)
        mask: broadcastable bool tensor — True = MASK OUT that position
    Returns:
        output:  (..., seq_q, d_v)
        weights: (..., seq_q, seq_k)  — useful for visualisation
    """
    d_k = Q.size(-1)

    # ① Raw attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (..., q, k)

    # ② Apply mask — fill masked positions with -inf so softmax → ~0
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))

    # ③ Softmax over the key dimension
    weights = torch.softmax(scores, dim=-1)   # (..., q, k)

    # ④ Optional dropout on attention weights
    if dropout > 0.0:
        weights = torch.nn.functional.dropout(weights, p=dropout)

    # ⑤ Weighted sum of values
    output = torch.matmul(weights, V)         # (..., q, d_v)
    return output, weights


# ── Helper: causal (look-ahead) mask ─────────────────────────────────────────
def causal_mask(size: int, device=torch.device("cpu")) -> torch.Tensor:
    """
    Upper-triangular mask (excluding diagonal).
    Shape: (1, size, size) — True = block that position.
    """
    mask = torch.triu(torch.ones(size, size, dtype=torch.bool, device=device), diagonal=1)
    return mask.unsqueeze(0)   # add batch dim for broadcasting


# ── Demo ──────────────────────────────────────────────────────────────────────
B, S, D = 2, 6, 32   # batch, seq_len, head_dim
Q = torch.randn(B, S, D)
K = torch.randn(B, S, D)
V = torch.randn(B, S, D)

mask = causal_mask(S)
print("Causal mask (6×6):\n", mask[0].int())

out, attn_w = scaled_dot_product_attention(Q, K, V, mask=mask)
print(f"\nOutput shape  : {out.shape}")
print(f"Weights shape : {attn_w.shape}")
print(f"Weights sum   : {attn_w[0].sum(dim=-1)}")   # should be 1.0 for unmasked rows


### 🏋️ Exercises — Section 7: Scaled Dot-Product Attention

**E7.1** Extend `scaled_dot_product_attention` to support a **key padding mask** that blocks out pad tokens. The mask should be a `(batch, seq_k)` bool tensor, and should be broadcast correctly into the `(batch, seq_q, seq_k)` score matrix.

**E7.2** Without changing `scaled_dot_product_attention`, build a **sliding window attention** mask for sequence length 8 and window size 3 — position `i` can only attend to positions `[i-1, i, i+1]`. Print the resulting `(8, 8)` mask.

**E7.3** Verify numerically that attention **temperature scaling** matters: compute attention weights for `d_k = 1`, `64`, `512` on the same Q and K. Print the entropy of the weight distribution for each — higher `d_k` without scaling leads to more peaked (lower entropy) distributions.

**E7.4** Implement **attention with a bias term** — add a `(seq_q, seq_k)` learnable matrix to the scores before softmax (used in T5 and ALiBi). Wrap it as a small module.


In [ ]:
# ── E7.1 — Key padding mask ──────────────────────────────────────────────────
def attention_with_key_padding(Q, K, V, key_padding_mask=None):
    """
    key_padding_mask: (batch, seq_k) bool — True = pad token (mask out)
    """
    # YOUR CODE HERE: expand key_padding_mask to (batch, seq_q, seq_k) and call
    # scaled_dot_product_attention
    pass

# Test
B, S, D = 2, 6, 16
Q = torch.randn(B, S, D); K = torch.randn(B, S, D); V = torch.randn(B, S, D)
key_mask = torch.tensor([[False]*4 + [True]*2,   # last 2 tokens are PAD in sample 0
                          [False]*6])             # no PAD in sample 1
# out, w = attention_with_key_padding(Q, K, V, key_mask)
# print("Padding mask attention output:", out.shape)

# ── E7.2 — Sliding window mask ────────────────────────────────────────────────
def sliding_window_mask(seq_len: int, window: int) -> torch.Tensor:
    """True = block that position. position i can attend to [i-w//2 ... i+w//2]"""
    # YOUR CODE HERE
    pass

# mask = sliding_window_mask(8, 3)
# print("Window mask (8x8):\n", mask.int())

# ── E7.3 — Temperature scaling experiment ────────────────────────────────────
def attention_entropy(d_k: int, apply_scale: bool = True):
    Q = torch.randn(1, 4, d_k)
    K = torch.randn(1, 4, d_k)
    V = torch.randn(1, 4, d_k)
    scale = math.sqrt(d_k) if apply_scale else 1.0
    scores = (Q @ K.transpose(-2, -1)) / scale
    weights = torch.softmax(scores, dim=-1)
    entropy = -(weights * (weights + 1e-9).log()).sum(-1).mean()
    return entropy.item()

# for d in [1, 64, 512]:
#     print(f"d_k={d:>4} | with scaling: {attention_entropy(d, True):.3f}"
#           f" | without: {attention_entropy(d, False):.3f}")

# ── E7.4 — Attention with learnable bias ─────────────────────────────────────
class BiasedAttention(nn.Module):
    def __init__(self, max_len: int):
        super().__init__()
        # YOUR CODE HERE: learnable (max_len, max_len) bias
        self.bias = nn.Parameter(torch.zeros(max_len, max_len))

    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        seq = Q.size(-2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
        scores = scores + self.bias[:seq, :seq]   # add learned bias
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        return torch.softmax(scores, dim=-1) @ V


---
<a id="8"></a>
## Section 8 — Multi-Head Attention

Instead of computing attention once with $d_{model}$-dimensional projections, Multi-Head Attention runs $h$ attention operations **in parallel** on smaller $d_k = d_{model}/h$ dimensional projections. Each head can learn to attend to different relationships.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\,W^O$$

$$\text{where } \text{head}_i = \text{Attention}(QW_i^Q,\, KW_i^K,\, VW_i^V)$$

**Implementation insight:** rather than creating $h$ separate linear layers, we project to a joint `(batch, seq, h * d_k)` tensor, reshape it to `(batch, h, seq, d_k)`, and run attention in bulk.


In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head self/cross-attention.

    Args:
        d_model  : total model dimension (e.g. 512)
        num_heads: number of attention heads h
        dropout  : dropout probability on attention weights
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model    = d_model
        self.num_heads  = num_heads
        self.d_k        = d_model // num_heads   # per-head dimension
        self.dropout    = dropout

        # Single linear for Q, K, V saves a bit of overhead
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(B, S, d_model) → (B, h, S, d_k)"""
        B, S, _ = x.shape
        x = x.view(B, S, self.num_heads, self.d_k)
        return x.transpose(1, 2)   # (B, h, S, d_k)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(B, h, S, d_k) → (B, S, d_model)"""
        B, _, S, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, S, self.d_model)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # Linear projections
        Q = self._split_heads(self.W_q(query))   # (B, h, S_q, d_k)
        K = self._split_heads(self.W_k(key))     # (B, h, S_k, d_k)
        V = self._split_heads(self.W_v(value))   # (B, h, S_k, d_k)

        # Expand mask for heads dim if provided
        if mask is not None and mask.dim() == 3:
            mask = mask.unsqueeze(1)   # (B, 1, S_q, S_k)

        # Scaled dot-product attention on all heads simultaneously
        attn_out, attn_w = scaled_dot_product_attention(Q, K, V, mask=mask, dropout=self.dropout)

        # Merge heads and apply output projection
        out = self.W_o(self._merge_heads(attn_out))  # (B, S_q, d_model)
        return out, attn_w


# ── Demo ──────────────────────────────────────────────────────────────────────
D_MODEL   = 128
NUM_HEADS = 8
S         = SEQ_LEN - 1   # 31

mha   = MultiHeadAttention(D_MODEL, NUM_HEADS, dropout=0.1)
x_seq = torch.randn(BATCH_SIZE, S, D_MODEL)
mask  = causal_mask(S)

out, weights = mha(x_seq, x_seq, x_seq, mask=mask)  # self-attention
print(f"MHA output shape    : {out.shape}")           # (8, 31, 128)
print(f"Attention weights   : {weights.shape}")       # (8, 8, 31, 31)

total = sum(p.numel() for p in mha.parameters())
print(f"MHA parameters      : {total:,}")


### 🏋️ Exercises — Section 8: Multi-Head Attention

**E8.1** Add a **dropout on the output projection** to `MultiHeadAttention` — apply dropout after `W_o` but before returning. Verify that training mode and eval mode produce different outputs.

**E8.2** Implement **Grouped Query Attention (GQA)** — used in LLaMA 2/3. Instead of `h` independent K, V projections, have only `g` groups (`g < h`). Each group of `h // g` query heads shares the same K and V. Modify `MultiHeadAttention` to accept a `kv_heads` argument.

**E8.3** Extract and visualise the attention weight matrix for a sample input: run a forward pass through `MultiHeadAttention`, retrieve `attn_w` of shape `(batch, heads, seq, seq)`, and print (or plot) head 0's weights for batch item 0.

**E8.4** Verify that **causal masking is working correctly**: for each position `i`, confirm that `attn_w[0, 0, i, j] == 0` for all `j > i` after a forward pass with a causal mask.


In [ ]:
# ── E8.1 — Output projection dropout ─────────────────────────────────────────
# Modify MultiHeadAttention to add nn.Dropout after W_o
# Then verify: mha.train() → random outputs; mha.eval() → deterministic outputs

# ── E8.2 — Grouped Query Attention (GQA) ─────────────────────────────────────
class GroupedQueryAttention(nn.Module):
    """
    GQA: num_heads query heads share kv_heads key/value heads.
    num_heads must be divisible by kv_heads.
    """

    def __init__(self, d_model: int, num_heads: int, kv_heads: int, dropout: float = 0.0):
        super().__init__()
        assert num_heads % kv_heads == 0
        self.num_heads = num_heads
        self.kv_heads  = kv_heads
        self.d_k       = d_model // num_heads
        self.groups    = num_heads // kv_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        # YOUR CODE HERE: W_k and W_v should project to kv_heads * d_k dimensions
        self.W_k = nn.Linear(d_model, kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, query, key, value, mask=None):
        B, S, _ = query.shape
        # Project Q: (B, S, h, d_k) → (B, h, S, d_k)
        Q = self.W_q(query).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        # Project K, V: (B, S, kv_heads, d_k) → repeat to num_heads
        K = self.W_k(key).view(B, S, self.kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(B, S, self.kv_heads, self.d_k).transpose(1, 2)
        # YOUR CODE HERE: repeat K and V along head dim to match num_heads
        # K = K.repeat_interleave(self.groups, dim=1)
        # V = V.repeat_interleave(self.groups, dim=1)
        out, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        out = out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.W_o(out)

# Test GQA: 8 query heads, 2 KV heads
# gqa = GroupedQueryAttention(D_MODEL, num_heads=8, kv_heads=2)
# x = torch.randn(BATCH_SIZE, S, D_MODEL)
# print("GQA output:", gqa(x, x, x).shape)

# ── E8.3 — Visualise attention weights ───────────────────────────────────────
mha   = MultiHeadAttention(D_MODEL, NUM_HEADS, dropout=0.0)
x_seq = torch.randn(1, 8, D_MODEL)
mask  = causal_mask(8)
with torch.no_grad():
    _, weights = mha(x_seq, x_seq, x_seq, mask=mask)

print("Attention weight matrix (head 0):")
print(weights[0, 0].round(decimals=3))

# ── E8.4 — Verify causal masking ─────────────────────────────────────────────
# For every row i, entries j > i should be 0
upper_triangle = weights[0, 0].triu(diagonal=1)
print(f"\nMax value above diagonal: {upper_triangle.max().item():.6f}  (should be ~0)")
assert upper_triangle.max().item() < 1e-6, "Causal mask is not working!"
print("Causal mask verified ✓")


---
<a id="9"></a>
## Section 9 — Feed-Forward Networks and Layer Normalization

Each Transformer block applies a **Position-wise FFN** identically to every position:

$$\text{FFN}(x) = \max(0,\; xW_1 + b_1)\,W_2 + b_2$$

The inner dimension is typically $4 \times d_{model}$.

**Layer Normalization** (vs Batch Norm):
- Normalizes over the *feature* dimension, not the *batch* dimension
- Works identically at train and inference time — no running statistics
- **Pre-LN** vs **Post-LN**: modern Transformers (GPT-2, LLaMA) apply LN _before_ each sub-layer (Pre-LN) for more stable training

**Residual (Add & Norm) pattern:**
```
output = LayerNorm(x + SubLayer(x))   # Post-LN (original paper)
output = x + SubLayer(LayerNorm(x))   # Pre-LN  (modern default)
```


In [ ]:
class PositionwiseFFN(nn.Module):
    """
    Two-layer feed-forward network applied independently at each position.
    Optionally uses GELU instead of ReLU (used in BERT, GPT-2).
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1, activation: str = "relu"):
        super().__init__()
        act = nn.ReLU() if activation == "relu" else nn.GELU()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            act,
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class AddNorm(nn.Module):
    """
    Pre-LN residual connection:  x + SubLayer(LayerNorm(x))
    Used in modern Transformers for better gradient flow.
    """

    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.norm    = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, sub_layer_out: torch.Tensor) -> torch.Tensor:
        # We apply norm to x before passing to sublayer (Pre-LN).
        # The sublayer result is passed in from outside for flexibility.
        return x + self.dropout(sub_layer_out)


# ── Demo ──────────────────────────────────────────────────────────────────────
D_FF = D_MODEL * 4   # 512

ffn  = PositionwiseFFN(D_MODEL, D_FF, dropout=0.1, activation="gelu")
norm = nn.LayerNorm(D_MODEL)
x    = torch.randn(BATCH_SIZE, S, D_MODEL)

# Pre-LN pattern
ffn_out = ffn(norm(x))          # sublayer sees normalized input
output  = x + ffn_out           # residual after sublayer

print(f"FFN output shape  : {ffn_out.shape}")
print(f"After Add & Norm  : {output.shape}")
print(f"Output mean ≈ 0?  : {output.mean().item():.4f}")

total_ffn = sum(p.numel() for p in ffn.parameters())
print(f"FFN parameters    : {total_ffn:,}")


### 🏋️ Exercises — Section 9: FFN and Layer Normalization

**E9.1** Implement **SwiGLU** — the activation used in LLaMA/PaLM: `SwiGLU(x) = SiLU(W₁x) ⊙ W₂x` where ⊙ is element-wise multiply. Build a `SwiGLUFFN` module and verify its output shape matches a standard FFN.

**E9.2** Implement **RMSNorm** — used in LLaMA instead of LayerNorm: $\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}} \cdot \gamma$. Compare its output to `nn.LayerNorm` on the same input.

**E9.3** Verify the Pre-LN vs Post-LN gradient flow difference: build two small 4-layer stacked FFN blocks (one Pre-LN, one Post-LN). Run a forward and backward pass on random input and compare the **gradient norm at layer 1** relative to **layer 4** for each.

**E9.4** Drop in `nn.GELU` and `nn.SiLU` as `PositionwiseFFN` activations. Benchmark all three (`ReLU`, `GELU`, `SiLU`) for 100 forward passes on a `(32, 128, 512)` input and print average time.


In [ ]:
# ── E9.1 — SwiGLU FFN ────────────────────────────────────────────────────────
class SwiGLUFFN(nn.Module):
    """
    SwiGLU: output = SiLU(W1(x)) * W2(x), then project with W3.
    To keep the same param count as a standard FFN with inner dim d_ff,
    use inner dim = 2/3 * d_ff (a common practice).
    """
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        inner = int(2 * d_ff / 3)
        # YOUR CODE HERE
        # self.W1 = nn.Linear(d_model, inner, bias=False)
        # self.W2 = nn.Linear(d_model, inner, bias=False)
        # self.W3 = nn.Linear(inner, d_model, bias=False)

    def forward(self, x):
        # YOUR CODE HERE
        # return self.W3(F.silu(self.W1(x)) * self.W2(x))
        pass

# swiglu = SwiGLUFFN(D_MODEL, D_FF)
# out = swiglu(torch.randn(4, 16, D_MODEL))
# print("SwiGLU output:", out.shape)

# ── E9.2 — RMSNorm ───────────────────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE: normalise by RMS and multiply by gamma
        pass

x_norm = torch.randn(4, 16, D_MODEL)
rms  = RMSNorm(D_MODEL)
ln   = nn.LayerNorm(D_MODEL)
# Outputs won't be identical but should both have similar scale
# print("RMSNorm output mean:", rms(x_norm).mean().item())
# print("LayerNorm output mean:", ln(x_norm).mean().item())

# ── E9.3 — Pre-LN vs Post-LN gradient norms ──────────────────────────────────
# YOUR CODE HERE
# Build 4-layer stacks for each variant
# Run backward and print grad norms of the first linear weight in layer 0 vs layer 3

# ── E9.4 — Activation benchmark ──────────────────────────────────────────────
import time as _time

x_bench = torch.randn(32, 128, 512)
for act_name in ["relu", "gelu", "silu"]:
    act_fn = {"relu": F.relu, "gelu": F.gelu, "silu": F.silu}[act_name]
    t0 = _time.perf_counter()
    for _ in range(100):
        _ = act_fn(x_bench)
    print(f"{act_name}: {(_time.perf_counter()-t0)*10:.2f} ms / pass")


---
<a id="10"></a>
## Section 10 — Encoder and Decoder Blocks

### Encoder Block
```
┌─────────────────────────────┐
│  x ──► LayerNorm ──► MHA ──► + ──► x'
│  x' ──► LayerNorm ──► FFN ──► + ──► output
└─────────────────────────────┘
```

### Decoder Block (adds Cross-Attention)
```
┌──────────────────────────────────────────────────────────┐
│  x  ──► LayerNorm ──► Masked MHA (causal) ──► + ──► x'
│  x' ──► LayerNorm ──► Cross MHA (K,V from encoder) ──► + ──► x''
│  x'' ──► LayerNorm ──► FFN ──► + ──► output
└──────────────────────────────────────────────────────────┘
```


In [ ]:
class EncoderBlock(nn.Module):
    """One Transformer encoder layer (Pre-LN)."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn        = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1      = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: torch.Tensor | None = None) -> torch.Tensor:
        # Pre-LN self-attention + residual
        attn_out, _ = self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), mask=src_mask)
        x = x + self.dropout(attn_out)

        # Pre-LN FFN + residual
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class DecoderBlock(nn.Module):
    """One Transformer decoder layer with masked self-attention + cross-attention (Pre-LN)."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)  # causal
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)  # encoder output
        self.ffn        = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1      = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.norm3      = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        enc_out: torch.Tensor,
        tgt_mask: torch.Tensor | None = None,
        src_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # 1. Masked self-attention (causal)
        n1 = self.norm1(x)
        sa_out, _ = self.self_attn(n1, n1, n1, mask=tgt_mask)
        x = x + self.dropout(sa_out)

        # 2. Cross-attention: query from decoder, key/value from encoder
        n2 = self.norm2(x)
        ca_out, _ = self.cross_attn(n2, enc_out, enc_out, mask=src_mask)
        x = x + self.dropout(ca_out)

        # 3. FFN
        x = x + self.dropout(self.ffn(self.norm3(x)))
        return x


# ── Stacked Encoder & Decoder ─────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.layers  = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm    = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


class Decoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.layers = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm   = nn.LayerNorm(d_model)

    def forward(self, x, enc_out, tgt_mask=None, src_mask=None):
        for layer in self.layers:
            x = layer(x, enc_out, tgt_mask, src_mask)
        return self.norm(x)


# ── Quick shape check ─────────────────────────────────────────────────────────
N_LAYERS = 3
enc = Encoder(N_LAYERS, D_MODEL, NUM_HEADS, D_FF, dropout=0.1)
dec = Decoder(N_LAYERS, D_MODEL, NUM_HEADS, D_FF, dropout=0.1)

src_seq = torch.randn(BATCH_SIZE, S, D_MODEL)   # encoder input
tgt_seq = torch.randn(BATCH_SIZE, S, D_MODEL)   # decoder input

enc_out = enc(src_seq)
print(f"Encoder output    : {enc_out.shape}")

tgt_mask = causal_mask(S)
dec_out  = dec(tgt_seq, enc_out, tgt_mask=tgt_mask)
print(f"Decoder output    : {dec_out.shape}")


### 🏋️ Exercises — Section 10: Encoder and Decoder Blocks

**E10.1** Modify `EncoderBlock` to add **stochastic depth** (DropPath): with probability `p`, skip the entire residual sub-layer during training (set the sub-layer output to zero). This is used in DeiT and many vision Transformers.

**E10.2** Build a **GPT-style decoder-only block**: remove cross-attention from `DecoderBlock` — it should only have masked self-attention and FFN. Stack 4 of these and run a forward pass.

**E10.3** Count and compare the number of parameters in an `EncoderBlock` vs `DecoderBlock` for `d_model=256, num_heads=8, d_ff=1024`. Verify analytically: the decoder should have ~50% more params (due to cross-attention).

**E10.4** Implement **layer-wise learning rate decay** for a stacked Encoder: assign LR multipliers `λ^(N-l)` to layer `l` (where `λ = 0.75`, `N` = total layers). Pass per-layer parameter groups to `AdamW`.


In [ ]:
# ── E10.1 — Stochastic Depth / DropPath ──────────────────────────────────────
class DropPath(nn.Module):
    """Randomly zeroes the entire residual branch during training."""
    def __init__(self, drop_prob: float = 0.1):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_prob == 0.0:
            return x
        # YOUR CODE HERE: with probability drop_prob, return zeros; else return x
        # Hint: sample a Bernoulli scalar: keep = torch.bernoulli(...)
        pass

# ── E10.2 — GPT-style decoder-only block ─────────────────────────────────────
class GPTBlock(nn.Module):
    """Decoder-only block: masked self-attention + FFN (no cross-attention)."""
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # YOUR CODE HERE — reuse MultiHeadAttention + PositionwiseFFN
        pass

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        # YOUR CODE HERE
        pass

# Stack 4 GPTBlocks and run forward
# gpt_blocks = nn.Sequential(*[GPTBlock(D_MODEL, NUM_HEADS, D_FF) for _ in range(4)])
# But Sequential doesn't pass mask — either use a loop or a wrapper
# x = torch.randn(BATCH_SIZE, S, D_MODEL)
# for block in gpt_blocks:
#     x = block(x, causal_mask(S))
# print("GPT stack output:", x.shape)

# ── E10.3 — Parameter count comparison ───────────────────────────────────────
d, h, ff = 256, 8, 1024
enc_block = EncoderBlock(d, h, ff)
dec_block = DecoderBlock(d, h, ff)

enc_params = sum(p.numel() for p in enc_block.parameters())
dec_params = sum(p.numel() for p in dec_block.parameters())
print(f"EncoderBlock params : {enc_params:,}")
print(f"DecoderBlock params : {dec_params:,}")
print(f"Ratio               : {dec_params/enc_params:.2f}x  (expected ~1.5x)")

# ── E10.4 — Layer-wise LR decay ───────────────────────────────────────────────
def layer_wise_lr_groups(encoder: Encoder, base_lr: float, decay: float = 0.75):
    """Returns parameter groups with decaying LRs from last layer to first."""
    groups = []
    N = len(encoder.layers)
    for l, layer in enumerate(encoder.layers):
        lr = base_lr * (decay ** (N - 1 - l))
        groups.append({"params": layer.parameters(), "lr": lr})
    return groups

# enc = Encoder(4, D_MODEL, NUM_HEADS, D_FF, dropout=0.1)
# param_groups = layer_wise_lr_groups(enc, base_lr=1e-3)
# for i, g in enumerate(param_groups):
#     print(f"Layer {i} lr: {g['lr']:.5f}")
# opt = optim.AdamW(param_groups)


---
<a id="11"></a>
## Section 11 — Full Transformer Architecture

Now we assemble every piece into a single `Transformer` module:

```
src tokens ──► TokenEmbedding ──► PositionalEncoding ──► Encoder ───┐
                                                                      ├──► Decoder ──► Linear ──► logits
tgt tokens ──► TokenEmbedding ──► PositionalEncoding ──────────────┘
```

This matches the original **"Attention is All You Need"** architecture and is the foundation for models like BERT (encoder-only), GPT (decoder-only), T5/BART (encoder-decoder).


In [ ]:
class Transformer(nn.Module):
    """
    Full encoder-decoder Transformer.

    Args:
        src_vocab_size : source vocabulary size
        tgt_vocab_size : target vocabulary size
        d_model        : embedding / hidden dimension
        num_heads      : number of attention heads
        num_layers     : number of encoder + decoder layers each
        d_ff           : inner dimension of FFN
        max_len        : maximum sequence length for positional encoding
        dropout        : dropout probability
    """

    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int   = 512,
        num_heads: int = 8,
        num_layers: int = 6,
        d_ff: int      = 2048,
        max_len: int   = 5000,
        dropout: float = 0.1,
    ):
        super().__init__()

        # ── Input / Output Embeddings ────────────────────────────────────────
        self.src_embed = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_embed = TokenEmbedding(tgt_vocab_size, d_model)
        self.src_pe    = PositionalEncoding(d_model, max_len, dropout)
        self.tgt_pe    = PositionalEncoding(d_model, max_len, dropout)

        # ── Encoder & Decoder ────────────────────────────────────────────────
        self.encoder = Encoder(num_layers, d_model, num_heads, d_ff, dropout)
        self.decoder = Decoder(num_layers, d_model, num_heads, d_ff, dropout)

        # ── Output Projection ────────────────────────────────────────────────
        self.output_proj = nn.Linear(d_model, tgt_vocab_size)

        # Weight tying: share weights between target embedding and output projection
        # (reduces parameters and often improves performance — used in GPT, LLaMA, etc.)
        self.output_proj.weight = self.tgt_embed.embed.weight

        self._init_weights()

    def _init_weights(self):
        """Xavier uniform initialization — standard for Transformer weights."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src: torch.Tensor, src_mask=None) -> torch.Tensor:
        return self.encoder(self.src_pe(self.src_embed(src)), src_mask)

    def decode(self, tgt: torch.Tensor, enc_out: torch.Tensor, tgt_mask=None, src_mask=None):
        return self.decoder(self.tgt_pe(self.tgt_embed(tgt)), enc_out, tgt_mask, src_mask)

    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor,
        src_mask=None,
        tgt_mask=None,
    ) -> torch.Tensor:
        enc_out = self.encode(src, src_mask)                   # (B, S_src, d_model)
        dec_out = self.decode(tgt, enc_out, tgt_mask, src_mask) # (B, S_tgt, d_model)
        logits  = self.output_proj(dec_out)                    # (B, S_tgt, vocab)
        return logits


# ── Instantiate and run a forward pass ───────────────────────────────────────
transformer = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=N_LAYERS,
    d_ff=D_FF,
    max_len=SEQ_LEN,
    dropout=0.1,
).to(DEVICE)

# Dummy token sequences
src_tokens = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, S)).to(DEVICE)
tgt_tokens = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, S)).to(DEVICE)
tgt_mask   = causal_mask(S).to(DEVICE)

with torch.no_grad():
    logits = transformer(src_tokens, tgt_tokens, tgt_mask=tgt_mask)

print(f"Logits shape      : {logits.shape}")    # (8, 31, 256)

total_params = sum(p.numel() for p in transformer.parameters())
print(f"Total parameters  : {total_params:,}")


### 🏋️ Exercises — Section 11: Full Transformer Architecture

**E11.1** Build a **GPT-2 small equivalent** using the `Transformer` class: `d_model=768, num_heads=12, num_layers=12, d_ff=3072, vocab_size=50257`. Count the total parameters and compare to the published GPT-2 small parameter count (~117M).

**E11.2** Implement a simple **greedy decoding** function for the Transformer: given a source sequence, encode it once, then generate target tokens one at a time by picking `argmax(logits[:, -1, :])` until an EOS token or `max_len` is reached.

**E11.3** Implement **beam search** with `beam_size=3`. At each step, maintain the top-k running sequences by cumulative log-probability and expand them.

**E11.4** Add a `src_key_padding_mask` to the full Transformer's `forward` so that padding tokens in `src` are ignored in the encoder self-attention and in the decoder cross-attention.


In [ ]:
# ── E11.1 — GPT-2 small parameter count ──────────────────────────────────────
# YOUR CODE HERE
# gpt2_small = Transformer(
#     src_vocab_size=50257, tgt_vocab_size=50257,
#     d_model=768, num_heads=12, num_layers=12, d_ff=3072
# )
# params = sum(p.numel() for p in gpt2_small.parameters())
# print(f"GPT-2 small params: {params/1e6:.1f}M  (published: ~117M)")

# ── E11.2 — Greedy decoding ───────────────────────────────────────────────────
EOS_TOKEN = VOCAB_SIZE - 1

@torch.no_grad()
def greedy_decode(model, src, max_len=30, sos_token=1):
    model.eval()
    src = src.unsqueeze(0).to(DEVICE)        # (1, S_src)
    enc_out = model.encode(src)              # encode once

    # Start with SOS token
    tgt_seq = torch.tensor([[sos_token]], device=DEVICE)  # (1, 1)

    for _ in range(max_len):
        mask   = causal_mask(tgt_seq.size(1)).to(DEVICE)
        dec_out = model.decode(tgt_seq, enc_out, tgt_mask=mask)
        logits  = model.output_proj(dec_out)   # (1, t, vocab)
        next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # (1, 1)
        tgt_seq  = torch.cat([tgt_seq, next_tok], dim=1)
        if next_tok.item() == EOS_TOKEN:
            break

    return tgt_seq.squeeze(0).tolist()

# Try greedy decoding
sample_src = torch.randint(0, VOCAB_SIZE, (S,))
# decoded = greedy_decode(transformer, sample_src)
# print("Greedy decode output:", decoded)

# ── E11.3 — Beam search (beam_size=3) ────────────────────────────────────────
@torch.no_grad()
def beam_search(model, src, beam_size=3, max_len=30, sos_token=1):
    model.eval()
    # YOUR CODE HERE
    # Each beam state: (log_prob, token_sequence)
    # beams = [(0.0, [sos_token])]
    # for step in range(max_len):
    #     all_candidates = []
    #     for log_p, seq in beams:
    #         ... expand each beam ...
    #     beams = sorted(all_candidates)[-beam_size:]
    # return best beam
    pass

# ── E11.4 — Source padding mask ───────────────────────────────────────────────
# Modify Transformer.forward to accept src_key_padding_mask
# and pass it through to the encoder self-attention and decoder cross-attention
# YOUR CODE HERE (hint: reshape mask for the MultiHeadAttention interface)


---
<a id="12"></a>
## Section 12 — Training Loop and Performance Optimization

A production-quality training loop needs four key additions beyond the basic step:

| Technique | Why |
|---|---|
| **Gradient clipping** (`clip_grad_norm_`) | Prevents exploding gradients common in deep Transformers |
| **LR Scheduler** | Warm-up → decay is critical for Transformer convergence (the original paper uses a custom schedule) |
| **`model.eval()` + `torch.no_grad()`** | Faster, memory-efficient validation |
| **Benchmarking GPU vs CPU** | Confirms device utilization and bottlenecks |


In [ ]:
import time

# ── Warmup + Cosine Decay LR Schedule ────────────────────────────────────────
class WarmupCosineScheduler(torch.optim.lr_scheduler.LambdaLR):
    """
    Linear warm-up for `warmup_steps` steps, then cosine decay to `min_lr_ratio`.
    The original Transformer paper uses a different formula but this is the
    modern practical default (used in GPT-3, LLaMA, etc.).
    """

    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr_ratio: float = 0.1):
        def lr_lambda(step: int) -> float:
            if step < warmup_steps:
                return step / max(1, warmup_steps)
            progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
            return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

        super().__init__(optimizer, lr_lambda)


# ── Training + Validation Helper Functions ────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scheduler, clip_norm=1.0):
    model.train()
    total_loss, n_batches = 0.0, 0

    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        # Decoder input = tgt[:-1],  labels = tgt[1:]
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        mask    = causal_mask(tgt_in.size(1)).to(DEVICE)

        # ── Forward ──────────────────────────────────────────────────────────
        optimizer.zero_grad()
        logits = model(src, tgt_in, tgt_mask=mask)          # (B, S-1, vocab)

        # CrossEntropyLoss expects (N, C, ...) and (N, ...)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        # ── Backward + clip ──────────────────────────────────────────────────
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, n_batches = 0.0, 0
    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        tgt_in   = tgt[:, :-1]
        tgt_out  = tgt[:, 1:]
        mask     = causal_mask(tgt_in.size(1)).to(DEVICE)
        logits   = model(src, tgt_in, tgt_mask=mask)
        loss     = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        total_loss += loss.item()
        n_batches  += 1
    return total_loss / n_batches


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
EPOCHS      = 3
LR          = 3e-4
WARMUP_STEPS= 50

train_dataset = SequenceDataset(800, SEQ_LEN, VOCAB_SIZE)
val_dataset   = SequenceDataset(200, SEQ_LEN, VOCAB_SIZE)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

total_steps   = EPOCHS * len(train_loader)

model     = Transformer(VOCAB_SIZE, VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF, SEQ_LEN).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss(ignore_index=0)   # 0 = PAD token
scheduler = WarmupCosineScheduler(optimizer, WARMUP_STEPS, total_steps)

# ── Training Loop ─────────────────────────────────────────────────────────────
print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Val Loss':>8}  {'Time (s)':>9}  {'LR':>10}")
print("-" * 55)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
    val_loss   = evaluate(model, val_loader, criterion)
    elapsed    = time.time() - t0
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"{epoch:>5}  {train_loss:>10.4f}  {val_loss:>8.4f}  {elapsed:>9.2f}  {current_lr:>10.2e}")

print("\nTraining complete.")


### 🏋️ Exercises — Section 12: Training Loop and Performance Optimization

**E12.1** Implement **early stopping**: halt training if validation loss has not improved for `patience=3` consecutive epochs. Log how many epochs it took to stop.

**E12.2** Add **perplexity** as an evaluation metric: `perplexity = exp(avg_cross_entropy_loss)`. Log it alongside loss in the training table.

**E12.3** Profile the training loop using `torch.profiler.profile` for 5 steps. Print the top 5 most time-consuming operations (use `key_averages().table`).

**E12.4** Implement **cosine annealing with warm restarts** (`CosineAnnealingWarmRestarts`) from `torch.optim.lr_scheduler`. Compare total training loss over 10 epochs against the baseline `WarmupCosineScheduler`.


In [ ]:
import math as _math

# ── E12.1 — Early stopping ────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience: int = 3):
        self.patience   = patience
        self.best_loss  = float("inf")
        self.counter    = 0
        self.should_stop = False

    def step(self, val_loss: float) -> bool:
        """Returns True if training should stop."""
        # YOUR CODE HERE
        pass

# Usage example (uncomment to test):
# es = EarlyStopping(patience=3)
# for epoch in range(1, 20):
#     val_loss = ...  # obtain from evaluate()
#     if es.step(val_loss):
#         print(f"Early stopping at epoch {epoch}")
#         break

# ── E12.2 — Perplexity metric ────────────────────────────────────────────────
def compute_perplexity(avg_loss: float) -> float:
    """Perplexity = exp(cross-entropy loss)"""
    # YOUR CODE HERE
    pass

# Test
# for loss_val in [1.0, 2.3, 5.0]:
#     print(f"loss={loss_val:.1f}  →  perplexity={compute_perplexity(loss_val):.1f}")

# ── E12.3 — Profiling with torch.profiler ────────────────────────────────────
# YOUR CODE HERE
# from torch.profiler import profile, record_function, ProfilerActivity
# with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
#     for i, (src, tgt) in enumerate(train_loader):
#         if i >= 5: break
#         ...  # run a single training step
# print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=5))

# ── E12.4 — CosineAnnealingWarmRestarts ───────────────────────────────────────
# YOUR CODE HERE — compare WarmupCosineScheduler vs CosineAnnealingWarmRestarts
# over 10 training epochs and print per-epoch validation loss for both
# from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts


---
<a id="13"></a>
## Section 13 — Mixed Precision Training with `torch.cuda.amp`

**Why:** FP32 uses 4 bytes per value; FP16 uses 2 bytes. Half-precision arithmetic is 2—8× faster on modern NVIDIA GPUs (Tensor Cores). Mixed precision keeps FP32 for weights and loss accumulation (for numerical stability) while using FP16 for forward/backward (for speed).

**Two key components:**

| Component | Purpose |
|---|---|
| `torch.autocast(device_type, dtype)` | Context manager that automatically casts ops to FP16/BF16 |
| `torch.cuda.amp.GradScaler` | Scales loss upward to avoid FP16 underflow, then unscales before `optimizer.step()` |

> **BF16 vs FP16:** BF16 (brain float, same exponent range as FP32) is numerically more stable than FP16 and is the preferred dtype on A100/H100 GPUs and Apple Silicon.


In [ ]:
from torch.cuda.amp import GradScaler

def train_amp(model, loader, optimizer, criterion, scheduler, scaler, clip_norm=1.0):
    """
    Training step with Automatic Mixed Precision (AMP).
    Falls back to FP32 gracefully when running on CPU.
    """
    model.train()
    total_loss, n_batches = 0.0, 0
    use_amp = DEVICE.type == "cuda"

    # Determine best dtype: prefer bfloat16 on Ampere+ GPUs, else float16
    amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16

    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        mask    = causal_mask(tgt_in.size(1)).to(DEVICE)

        optimizer.zero_grad()

        # ── Autocast: ops inside here run in reduced precision ────────────────
        with torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=use_amp):
            logits = model(src, tgt_in, tgt_mask=mask)
            loss   = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        # ── Scaled backward ───────────────────────────────────────────────────
        scaler.scale(loss).backward()                          # scale loss to avoid underflow
        scaler.unscale_(optimizer)                             # unscale for clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        scaler.step(optimizer)                                 # update weights
        scaler.update()                                        # adjust scale factor
        scheduler.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / n_batches


# ── Benchmark FP32 vs AMP ──────────────────────────────────────────────────
def benchmark(use_mixed_precision: bool, n_steps: int = 20):
    m = Transformer(VOCAB_SIZE, VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF, SEQ_LEN).to(DEVICE)
    opt = optim.AdamW(m.parameters(), lr=LR)
    sch = WarmupCosineScheduler(opt, 5, n_steps)
    scaler = GradScaler(enabled=(use_mixed_precision and DEVICE.type == "cuda"))
    crit = nn.CrossEntropyLoss()

    loader = DataLoader(SequenceDataset(n_steps * BATCH_SIZE, SEQ_LEN, VOCAB_SIZE), batch_size=BATCH_SIZE)

    # warmup
    src, tgt = next(iter(loader))
    _ = m(src.to(DEVICE), tgt[:, :-1].to(DEVICE))

    start = time.perf_counter()
    train_amp(m, loader, opt, crit, sch, scaler)
    elapsed = time.perf_counter() - start

    return elapsed


print("Benchmarking (FP32 vs AMP)...")
t_fp32 = benchmark(use_mixed_precision=False)
t_amp  = benchmark(use_mixed_precision=True)

print(f"\n{'Mode':<20} {'Time (s)':>10}")
print("-" * 32)
print(f"{'FP32 (baseline)':<20} {t_fp32:>10.3f}")
print(f"{'AMP (FP16/BF16)':<20} {t_amp:>10.3f}")

if t_fp32 > t_amp:
    print(f"\nSpeedup: {t_fp32 / t_amp:.2f}×  (AMP faster)")
else:
    print(f"\nNote: No speedup on {DEVICE} — AMP benefits are GPU-specific (Tensor Cores).")


### 🏋️ Exercises — Section 13: Mixed Precision Training

**E13.1** Measure **peak GPU memory** (using `torch.cuda.max_memory_allocated()`) with FP32 vs AMP training for `batch_size=32`. Report the memory reduction ratio.

**E13.2** Experiment with `GradScaler` scale factor: print the current scale after each batch for 20 steps. Observe how it grows when no overflow occurs and shrinks when overflow is detected (force an overflow by temporarily scaling up the loss).

**E13.3** Rewrite `train_amp` to use `torch.compile(model)` (PyTorch 2.0+) and re-run the benchmark. Print speedup vs non-compiled AMP. *(Note: compilation takes ~30s the first time.)*

**E13.4** Implement a mixed-precision **inference** function that uses `torch.autocast` but no `GradScaler`. Time it against FP32 inference for 100 forward passes on a batch of size 16.


In [ ]:
# ── E13.1 — Memory usage: FP32 vs AMP ────────────────────────────────────────
def measure_peak_memory(use_amp: bool, batch_size: int = 32):
    if DEVICE.type != "cuda":
        print("GPU not available — skipping memory measurement")
        return
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    m = Transformer(VOCAB_SIZE, VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF, SEQ_LEN).to(DEVICE)
    opt = optim.AdamW(m.parameters(), lr=LR)
    src = torch.randint(0, VOCAB_SIZE, (batch_size, S)).to(DEVICE)
    tgt = torch.randint(0, VOCAB_SIZE, (batch_size, S)).to(DEVICE)
    mask = causal_mask(S).to(DEVICE)
    crit = nn.CrossEntropyLoss()
    use_amp_flag = use_amp and DEVICE.type == "cuda"
    amp_dtype = torch.bfloat16 if (use_amp_flag and torch.cuda.is_bf16_supported()) else torch.float16
    with torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=use_amp_flag):
        logits = m(src, tgt[:, :-1], tgt_mask=causal_mask(tgt.size(1)-1).to(DEVICE))
        loss = crit(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
    loss.backward()
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    print(f"{'AMP' if use_amp else 'FP32'} peak memory: {peak_mb:.1f} MB")
    return peak_mb

# mb_fp32 = measure_peak_memory(False)
# mb_amp  = measure_peak_memory(True)
# if mb_fp32 and mb_amp:
#     print(f"Memory reduction: {mb_fp32/mb_amp:.2f}x")

# ── E13.2 — GradScaler scale factor log ──────────────────────────────────────
# YOUR CODE HERE
# scaler = GradScaler()
# for step, (src, tgt) in enumerate(train_loader):
#     if step >= 20: break
#     ...  # run training step with scaler
#     print(f"step {step:>2}: scale = {scaler.get_scale()}")

# ── E13.3 — torch.compile + AMP ──────────────────────────────────────────────
# YOUR CODE HERE (PyTorch >= 2.0)
# compiled_model = torch.compile(transformer)
# t_compiled = benchmark_fn(compiled_model, use_amp=True)
# print(f"Compiled AMP: {t_compiled:.3f}s")

# ── E13.4 — Mixed-precision inference ────────────────────────────────────────
@torch.no_grad()
def amp_inference(model, src, tgt):
    use_amp = DEVICE.type == "cuda"
    dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
    with torch.autocast(device_type=DEVICE.type, dtype=dtype, enabled=use_amp):
        return model(src, tgt, tgt_mask=causal_mask(tgt.size(1)).to(DEVICE))

# Benchmark inference
src_i = torch.randint(0, VOCAB_SIZE, (16, S)).to(DEVICE)
tgt_i = torch.randint(0, VOCAB_SIZE, (16, S)).to(DEVICE)

# FP32 inference timing
t0 = time.perf_counter()
for _ in range(100):
    with torch.no_grad():
        _ = transformer(src_i, tgt_i, tgt_mask=causal_mask(S).to(DEVICE))
t_fp32 = time.perf_counter() - t0

# AMP inference timing
t0 = time.perf_counter()
for _ in range(100):
    _ = amp_inference(transformer, src_i, tgt_i)
t_amp = time.perf_counter() - t0

print(f"FP32 inference (100 passes): {t_fp32*1000:.1f} ms")
print(f"AMP  inference (100 passes): {t_amp*1000:.1f} ms")


---
<a id="14"></a>
## Section 14 — Model Checkpointing and Saving

**Always save `state_dict`** — not the whole model object — for portability across PyTorch versions and refactored code.

A complete checkpoint should contain:
```python
{
    "epoch"       : current epoch,
    "model_state" : model.state_dict(),
    "optim_state" : optimizer.state_dict(),
    "sched_state" : scheduler.state_dict(),
    "scaler_state": scaler.state_dict(),   # if using AMP
    "val_loss"    : best_val_loss,
}
```

This lets you **resume training exactly** where it left off — including the optimizer momentum buffers and the LR schedule position.


In [ ]:
import os

CHECKPOINT_DIR = "/teamspace/studios/this_studio/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def save_checkpoint(
    path: str,
    epoch: int,
    model: nn.Module,
    optimizer,
    scheduler,
    scaler,
    val_loss: float,
):
    """Save a full training checkpoint."""
    torch.save(
        {
            "epoch"       : epoch,
            "model_state" : model.state_dict(),
            "optim_state" : optimizer.state_dict(),
            "sched_state" : scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "val_loss"    : val_loss,
        },
        path,
    )
    print(f"  ✔ Checkpoint saved → {path}")


def load_checkpoint(path: str, model: nn.Module, optimizer=None, scheduler=None, scaler=None):
    """Load checkpoint and restore all states."""
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    if optimizer  : optimizer.load_state_dict(ckpt["optim_state"])
    if scheduler  : scheduler.load_state_dict(ckpt["sched_state"])
    if scaler     : scaler.load_state_dict(ckpt["scaler_state"])
    print(f"  ✔ Checkpoint loaded from {path}  (epoch {ckpt['epoch']}, val_loss {ckpt['val_loss']:.4f})")
    return ckpt["epoch"], ckpt["val_loss"]


# ── Best-model checkpointing loop ────────────────────────────────────────────
EPOCHS        = 3
best_val_loss = float("inf")

model     = Transformer(VOCAB_SIZE, VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF, SEQ_LEN).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss(ignore_index=0)
total_steps = EPOCHS * len(train_loader)
scheduler = WarmupCosineScheduler(optimizer, WARMUP_STEPS, total_steps)
scaler    = GradScaler(enabled=(DEVICE.type == "cuda"))

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Val Loss':>8}  {'Status':>12}")
print("-" * 46)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_amp(model, train_loader, optimizer, criterion, scheduler, scaler)
    val_loss   = evaluate(model, val_loader, criterion)

    status = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_path     = os.path.join(CHECKPOINT_DIR, "best_model.pt")
        save_checkpoint(ckpt_path, epoch, model, optimizer, scheduler, scaler, val_loss)
        status = "← best"

    print(f"{epoch:>5}  {train_loss:>10.4f}  {val_loss:>8.4f}  {status:>12}")


In [ ]:
# ── Restore from checkpoint and do inference ─────────────────────────────────
fresh_model = Transformer(VOCAB_SIZE, VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF, SEQ_LEN).to(DEVICE)
fresh_opt   = optim.AdamW(fresh_model.parameters(), lr=LR)
fresh_sch   = WarmupCosineScheduler(fresh_opt, WARMUP_STEPS, total_steps)
fresh_scaler= GradScaler(enabled=(DEVICE.type == "cuda"))

start_epoch, best_loss = load_checkpoint(
    os.path.join(CHECKPOINT_DIR, "best_model.pt"),
    fresh_model, fresh_opt, fresh_sch, fresh_scaler,
)
print(f"Resuming from epoch {start_epoch+1}")

# ── Inference-only: only state_dict needed ────────────────────────────────────
inference_model = Transformer(VOCAB_SIZE, VOCAB_SIZE, D_MODEL, NUM_HEADS, N_LAYERS, D_FF, SEQ_LEN).to(DEVICE)
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pt"), map_location=DEVICE)
inference_model.load_state_dict(ckpt["model_state"])
inference_model.eval()

test_src = torch.randint(0, VOCAB_SIZE, (1, S)).to(DEVICE)
test_tgt = torch.randint(0, VOCAB_SIZE, (1, S)).to(DEVICE)
with torch.no_grad():
    out = inference_model(test_src, test_tgt, tgt_mask=causal_mask(S).to(DEVICE))
print(f"\nInference output shape: {out.shape}")
print("Next-token probs (first position):", torch.softmax(out[0, 0], dim=-1).topk(5))


### 🏋️ Exercises — Section 14: Model Checkpointing and Saving

**E14.1** Extend `save_checkpoint` to also save the current **random states** (`torch.get_rng_state()`, `torch.cuda.get_rng_state()` if on GPU) so that training can be resumed with exactly reproducible results.

**E14.2** Implement a `ModelVersionManager` that stores the **last N checkpoints** (e.g. `N=3`) and automatically deletes older ones to save disk space. Test it over 6 epochs.

**E14.3** Export the Transformer to **ONNX** format using `torch.onnx.export`. Verify the exported model produces the same output as the PyTorch model on a test input (within floating point tolerance).

**E14.4** Implement **partial weight loading** — load only the encoder weights from a checkpoint into a new model that has a different decoder size. Use `strict=False` in `load_state_dict` and print which keys were missing or unexpected.


In [ ]:
import glob

# ── E14.1 — Save/restore RNG state ───────────────────────────────────────────
def save_checkpoint_v2(path, epoch, model, optimizer, scheduler, scaler, val_loss):
    """Extended checkpoint that also stores RNG states for exact reproducibility."""
    ckpt = {
        "epoch"       : epoch,
        "model_state" : model.state_dict(),
        "optim_state" : optimizer.state_dict(),
        "sched_state" : scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "val_loss"    : val_loss,
        # YOUR CODE HERE — add CPU and GPU rng states
        "rng_cpu"     : torch.get_rng_state(),
        # "rng_gpu"   : torch.cuda.get_rng_state() if DEVICE.type == "cuda" else None,
    }
    torch.save(ckpt, path)
    print(f"  ✔ v2 checkpoint saved → {path}")


# ── E14.2 — ModelVersionManager (keep last N) ─────────────────────────────────
class ModelVersionManager:
    """Keeps only the last `max_keep` checkpoints on disk."""

    def __init__(self, directory: str, max_keep: int = 3):
        self.directory = directory
        self.max_keep  = max_keep
        os.makedirs(directory, exist_ok=True)

    def save(self, epoch: int, model, optimizer, scheduler, scaler, val_loss: float):
        path = os.path.join(self.directory, f"ckpt_epoch{epoch:03d}.pt")
        save_checkpoint(path, epoch, model, optimizer, scheduler, scaler, val_loss)
        self._prune()

    def _prune(self):
        """Delete oldest checkpoints beyond max_keep."""
        ckpts = sorted(glob.glob(os.path.join(self.directory, "ckpt_epoch*.pt")))
        for old in ckpts[: -self.max_keep]:
            os.remove(old)
            print(f"  🗑  Deleted old checkpoint: {os.path.basename(old)}")

# Test over 6 epochs
# vm = ModelVersionManager(CHECKPOINT_DIR, max_keep=3)
# for epoch in range(1, 7):
#     vm.save(epoch, model, optimizer, scheduler, scaler, val_loss=5.0 - epoch * 0.5)

# ── E14.3 — ONNX export ───────────────────────────────────────────────────────
# YOUR CODE HERE
# onnx_path = os.path.join(CHECKPOINT_DIR, "transformer.onnx")
# src_dummy = torch.randint(0, VOCAB_SIZE, (1, S))
# tgt_dummy = torch.randint(0, VOCAB_SIZE, (1, S))
# mask_dummy = causal_mask(S)
# torch.onnx.export(
#     transformer.cpu(), (src_dummy, tgt_dummy, None, mask_dummy),
#     onnx_path, input_names=["src","tgt","src_mask","tgt_mask"],
#     output_names=["logits"], opset_version=14,
# )
# print(f"ONNX model saved to {onnx_path}")

# ── E14.4 — Partial weight loading ────────────────────────────────────────────
# Load encoder weights from a checkpoint into a differently-sized model
ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")

    # Build a model with a different number of decoder layers
    new_model = Transformer(VOCAB_SIZE, VOCAB_SIZE, D_MODEL, NUM_HEADS,
                             num_layers=2,   # different from saved (3 layers)
                             d_ff=D_FF, max_len=SEQ_LEN)

    result = new_model.load_state_dict(ckpt["model_state"], strict=False)
    print(f"\nPartial load results:")
    print(f"  Missing keys  ({len(result.missing_keys):>2}): {result.missing_keys[:3]} ...")
    print(f"  Unexpected    ({len(result.unexpected_keys):>2}): {result.unexpected_keys[:3]} ...")
else:
    print("Run Section 14 training cells first to generate a checkpoint.")


---

## Summary — What You Built

| Section | Component | Key PyTorch APIs |
|---|---|---|
| 1 | Tensors & Operations | `torch.tensor`, `matmul`, `einsum`, broadcasting |
| 2 | Autograd | `requires_grad`, `backward()`, `zero_grad()` |
| 3 | nn.Module | `nn.Linear`, `nn.Sequential`, `parameters()` |
| 4 | Optimizers & Losses | `AdamW`, `CrossEntropyLoss`, training step |
| 5 | Data Pipeline | `Dataset`, `DataLoader`, `__getitem__` |
| 6 | Input Representation | `nn.Embedding`, sinusoidal `PositionalEncoding` |
| 7 | Attention Mechanism | Scaled dot-product attention, causal mask |
| 8 | Multi-Head Attention | Head splitting, parallel attention, output projection |
| 9 | FFN & Normalization | `PositionwiseFFN`, `nn.LayerNorm`, Pre-LN residuals |
| 10 | Encoder & Decoder | Stacked blocks, cross-attention |
| 11 | Full Transformer | End-to-end assembly, weight tying, Xavier init |
| 12 | Training Loop | Gradient clipping, LR warm-up + cosine decay |
| 13 | Mixed Precision | `torch.autocast`, `GradScaler`, FP16/BF16 |
| 14 | Checkpointing | `state_dict`, save/load, best-model tracking |

### Next Steps
- **Decoder-only (GPT-style):** remove cross-attention, keep only the masked self-attention stack
- **Encoder-only (BERT-style):** remove decoder, use bidirectional attention, add MLM pre-training
- **Flash Attention:** drop-in replacement for `scaled_dot_product_attention` — 5–10× memory efficient
- **torch.compile:** `model = torch.compile(model)` for kernel fusion (PyTorch 2.0+)
- **Distributed training:** `torch.nn.parallel.DistributedDataParallel` (DDP) for multi-GPU scaling
